In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import Model,Input,layers
import tensorflow as tf

In [ ]:
base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

In [ ]:
block3_output = base.get_layer('block3_pool').output
block5_output = base.get_layer('block5_pool').output

In [ ]:
feat_model_3 = Model(inputs = base.input, outputs = block3_output)
feat_model_5 = Model(inputs = base.input, outputs = block5_output)

In [ ]:
import numpy as np
img = np.random.rand(1,224,224,3)
feats_3 = feat_model_3.predict(img)
feats_5 = feat_model_5.predict(img)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


In [ ]:
print(feats_3.shape)
print(feats_5.shape)

(1, 28, 28, 256)
(1, 7, 7, 512)


In [ ]:
multi_output_models = Model(
    inputs = base.inputs,
    outputs = [
        base.get_layer('block1_pool').output,
        base.get_layer('block3_pool').output,
        base.get_layer('block5_pool').output
    ]
)

In [ ]:
low, mid, high = multi_output_models.predict(img)
print(low.shape, mid.shape, high.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 817ms/step
(1, 112, 112, 64) (1, 28, 28, 256) (1, 7, 7, 512)


In [ ]:
# surgical freezing strategis
base = VGG16(weights = 'imagenet', include_top = False, input_shape=(224,224,3))
inputs = base.input
x = base.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
output = layers.Dense(1, activation='sigmoid')(x)
model = Model(inputs = inputs, outputs = output)


In [ ]:
# method 1: freezing every thing
#use : when datasets is tiny (<1000 images)
base.trainable = False

In [ ]:
# method 2: freezing specfic block
#use: medium datasets want to adapt high level features
base.trainble = True
for layers in base.layers:
  if layers.name.startswith('block5') or layers.name.startswith('block4'):
    layers.trainable = True
  else:
    layers.trainable = False

In [ ]:
# method3 ; freeze by index number
base.trainable = True
freeze_untill = 15
for i, layers in enumerate(base.layers):
  layers.trainable = (i>=freeze_untill)

In [ ]:
# always use small learning rate when unfreezing
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 1e-5),
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

In [ ]:
model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,846,273 (56.63 MB)

 Trainable params: 7,211,009 (27.51 MB)

 Non-trainable params: 7,635,264 (29.13 MB)

In [ ]:
trainable_count = sum(1 for l in model.layers if l.trainable)
frozen_count    = sum(1 for l in model.layers if not l.trainable)
print(f"Trainable: {trainable_count} | Frozen: {frozen_count}")

Trainable: 7 | Frozen: 15


In [ ]:
# debugging using block name
layer = model.get_layer('block5_conv1')
print(layer.name)          # block5_conv1
print(layer.output.shape)  # (None, 14, 14, 512)
print(layer.trainable)     # True or False
print(layer.count_params()) # number of weights

block5_conv1
(None, 14, 14, 512)
True
2359808


In [ ]:
# list all layers with shape
for layer in model.layers:
    print(f"layer:{layer}")
    print(f"{layer.name}\n {layer.output.shape}\n {layer.trainable}\n {layer.count_params()}")

layer:<InputLayer name=input_layer_2, built=True>
input_layer_2
 (None, 224, 224, 3)
 False
 0
layer:<Conv2D name=block1_conv1, built=True>
block1_conv1
 (None, 224, 224, 64)
 False
 1792
layer:<Conv2D name=block1_conv2, built=True>
block1_conv2
 (None, 224, 224, 64)
 False
 36928
layer:<MaxPooling2D name=block1_pool, built=True>
block1_pool
 (None, 112, 112, 64)
 False
 0
layer:<Conv2D name=block2_conv1, built=True>
block2_conv1
 (None, 112, 112, 128)
 False
 73856
layer:<Conv2D name=block2_conv2, built=True>
block2_conv2
 (None, 112, 112, 128)
 False
 147584
layer:<MaxPooling2D name=block2_pool, built=True>
block2_pool
 (None, 56, 56, 128)
 False
 0
layer:<Conv2D name=block3_conv1, built=True>
block3_conv1
 (None, 56, 56, 256)
 False
 295168
layer:<Conv2D name=block3_conv2, built=True>
block3_conv2
 (None, 56, 56, 256)
 False
 590080
layer:<Conv2D name=block3_conv3, built=True>
block3_conv3
 (None, 56, 56, 256)
 False
 590080
layer:<MaxPooling2D name=block3_pool, built=True>
block3_p

In [ ]:
# get weights of specfic layer
w, b = model.get_layer('dense').get_weights()
print(w.shape,b.shape)

(512, 256) (256,)


In [ ]:
# set weight manually(usefull for manually weight transfer)
model.get_layer('dense').set_weights([w,b])

In [ ]:
# ── build a debugger model — outputs EVERY layer ───────────────
debug_model = Model(
    inputs=model.input,
    outputs=[layer.output for layer in model.layers
             if hasattr(layer, 'output')]
)
all_outputs = debug_model.predict(img)
for layer, out in zip(model.layers, all_outputs):
    print(f"{layer.name}: {out.shape}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 909ms/step
input_layer_2: (1, 224, 224, 3)
block1_conv1: (1, 224, 224, 64)
block1_conv2: (1, 224, 224, 64)
block1_pool: (1, 112, 112, 64)
block2_conv1: (1, 112, 112, 128)
block2_conv2: (1, 112, 112, 128)
block2_pool: (1, 56, 56, 128)
block3_conv1: (1, 56, 56, 256)
block3_conv2: (1, 56, 56, 256)
block3_conv3: (1, 56, 56, 256)
block3_pool: (1, 28, 28, 256)
block4_conv1: (1, 28, 28, 512)
block4_conv2: (1, 28, 28, 512)
block4_conv3: (1, 28, 28, 512)
block4_pool: (1, 14, 14, 512)
block5_conv1: (1, 14, 14, 512)
block5_conv2: (1, 14, 14, 512)
block5_conv3: (1, 14, 14, 512)
block5_pool: (1, 7, 7, 512)
global_average_pooling2d: (1, 512)
dense: (1, 256)
dense_1: (1, 1)


In [ ]:
model.save('my_model.keras')                         # saves everything
model = tf.keras.models.load_model('my_model.keras') # restores exactly

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 22 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [44]:
# the two phase trainnig strategy  - used in production
#phase1 - train head only - fast convergence
base.trainable = False
model.compile(optimizer = tf.keras.optimizers.Adam(1e-3),
              loss = 'binary_crossentropy',
              metrics = ['accuracy'])
# model.fit(train_ds, epochs=10, validaton_data = val_ds)

#phase2 : unfreeze top blocks, fine-tune carefully
base.trainable = True
for layer in base.layers:
  layer.trainable = layer.name.startswith('block5')

# Must recompile after changing trainability

model.compile(optimizer = tf.keras.optimizers.Adam(1e-5),
              loss = 'binary_crossentropy',
              metrics = ['accuracy'])
#model.fit(train_ds, epochs = 10, validation_data = val_ds)